# Практика · Навчання трансформера

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє: [homework.html](homework.html)

> ⏱ Зошит навчає **33 маленькі мовні моделі**, робить два заміри часу й один — памʼяті.
> Заміряно: **близько 5 хвилин процесорного часу** (285 с) на чотирьох ядрах без
> відеокарти, в **один потік**. Стінного часу на вільній машині виходить
> приблизно стільки ж; на завантаженій — у кілька разів більше. Зошит друкує
> і навантаження машини, і свій власний процесорний час — останньою клітинкою.

## Задача зошита

У нас є корпус українських речень і трансформер, який учиться вгадувати наступне
слово. Питання, на яке ми відповідаємо, практичне й неприємне:

**скільки коштує цей трансформер — і що саме він за ці гроші купує?**

Відповідь ми складемо з чотирьох замірів:

1. **скільки коштує довжина.** Увага рахує всі пари позицій, тож вартість має
   рости як квадрат довжини. Перевіримо це годинником, а не формулою;
2. **скільки памʼяті зʼїдає довший вхід** — і чому кеш ключів та значень
   робить генерацію тексту принципово дешевшою;
3. **яку швидкість навчання брати.** Переберемо сітку й оберемо на **відкладеній**
   частині — тій, якої модель не бачить і яка не є перевірною;
4. **скільки якості купує кожне подвоєння даних.** Це головний замір: ми навчимо
   ту саму модель на 1/16, 1/8, 1/4, 1/2 і на всій купі речень, **давши кожній
   однакову кількість оновлень**, і побудуємо криву «якість від обсягу даних».
   Поруч поставимо згладжені лічильники — модель без жодного навчання.

Наприкінці ми екстраполюємо обидві криві й дістанемо число, заради якого все
робилося: **скільки тексту треба, щоб трансформер наздогнав лічильники.**

Дані справжні: це українські переклади інтерфейсів, які вже лежать у системі
читача. Нічого не завантажуємо.

## 0 · Середовище й чому ми фіксуємо потоки

Зошит міряє час, а час на спільній машині бреше двома різними способами.

**Стінний годинник** (той, що на стіні) показує, скільки минуло реального часу.
Якщо поруч рахується щось іще, він покаже більше — і число нічого не варте.

**Процесорний годинник** (`time.process_time()`) рахує лише той час, коли
процесор працював саме над нашою програмою. Він набагато чесніший — але має
власну пастку: якщо бібліотека лінійної алгебри розкладає роботу на кілька
потоків, то потоки, які **чекають** один на одного, теж рахуються як робота.
Спотворення сягає десятків разів.

Лікується це фіксацією потоків, і чотири рядки нижче мусять стояти **до** імпорту
`numpy` і `torch`: після імпорту бібліотека вже прочитала змінні середовища й на
зміну не зреагує.

In [ ]:
import os
# Ці чотири рядки — ДО імпорту numpy і torch. Інакше процесорний час
# рахуватиме ще й очікування потоків один на одного.
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import sys, re, glob, gettext, math, time, json, copy, gc
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(1)          # те саме для torch, уже після імпорту

notebook_started = time.process_time()   # з цієї миті рахуємо час усього зошита

print('python              ', sys.version.split()[0])
print('numpy               ', np.__version__)
print('torch               ', torch.__version__)
print('ядер у машині       ', os.cpu_count())
print('потоків у torch     ', torch.get_num_threads())
print('відеокарта          ', 'є' if torch.cuda.is_available() else 'немає')
print('навантаження машини ', round(os.getloadavg()[0], 2))

## 1 · Корпус

Беремо українські переклади інтерфейсів із `/usr/share/locale/uk/LC_MESSAGES/`.
Це справжня українська мова, вона вже лежить на машині, і вона однакова від
запуску до запуску — нічого не завантажуємо й нічого не генеруємо.

Ділити текст на слова будемо канонічним для курсу регулярним виразом. У ньому
апостроф — **звʼязка всередині слова**, тому `зʼєднання` лишається одним токеном,
а не розпадається на два.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"

def load_sentences():
    '''Читає каталоги перекладів і повертає список речень, поділених на слова.'''
    out = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                      # зламаний файл каталогу просто пропускаємо
        for source, target in catalog._catalog.items():
            # беремо лише справжні переклади: рядок довший за 30 символів
            # і не є службовим заголовком каталогу
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                out.append(re.findall(TOKEN_PATTERN, target.lower()))
    return out

sentences = load_sentences()
if len(sentences) < 1000:
    raise RuntimeError('українських каталогів у системі майже немає — '
                       'зошит без них не має на чому працювати')

lengths = np.array([len(s) for s in sentences if s])
print('речень у корпусі  ', len(sentences))
print('слововживань      ', int(lengths.sum()))
print('різних словоформ  ', len({w for s in sentences for w in s}))
print('медіана довжини   ', int(np.median(lengths)), 'слів')
print('90-й перцентиль   ', int(np.percentile(lengths, 90)), 'слів')
print('найдовше речення  ', int(lengths.max()), 'слів')

**Запамʼятай медіану.** Половина наших речень коротша за сім слів. Далі це
виявиться важливим двічі: квадратична вартість уваги на таких довжинах не коштує
майже нічого, а от **лічильникам** короткі речення дуже до вподоби.

## 2 · Три частини: навчальна, відкладена, перевірна

Ділити треба на **три** частини, а не на дві, і причина не формальна.

- **навчальна** — на ній модель учиться;
- **перевірна** — на ній ми оголошуємо результат. Модель не бачить її ніколи;
- **відкладена** — на ній ми **добираємо гіперпараметри** (швидкість навчання,
  сталу згладжування). Вона теж не бере участі в навчанні, але ми дивимось на неї
  багато разів.

Якби ми добирали швидкість навчання по перевірній частині, то підглядали б у
відповідь: обрали б те значення, яке пощастило на цьому конкретному тексті, і
кінцеве число вийшло б завищеним. Тому відкладена вибірка відрізається **від
навчальної частини**, а не від перевірної.

Спершу відсіюємо надто короткі й надто довгі речення: у реченні з одного слова
нема чого передбачати, а хвіст із півтисячі слів роздув би батчі заповнювачем.

In [ ]:
kept = [s for s in sentences if 2 <= len(s) <= 30]

rng = np.random.default_rng(0)
order = rng.permutation(len(kept))
n_test = len(kept) // 10
test_positions = set(order[:n_test].tolist())

train_all = [kept[i] for i in range(len(kept)) if i not in test_positions]
test_sents = [kept[i] for i in range(len(kept)) if i in test_positions]

# відкладена — 5 % навчальної частини, окреме зерно
rng_dev = np.random.default_rng(1)
order_dev = rng_dev.permutation(len(train_all))
n_dev = int(round(len(train_all) * 0.05))
dev_positions = set(order_dev[:n_dev].tolist())

dev_sents = [train_all[i] for i in range(len(train_all)) if i in dev_positions]
train_sents = [train_all[i] for i in range(len(train_all)) if i not in dev_positions]

print('після відсіву 2..30 слів:', len(kept), 'речень')
print('навчальних  ', len(train_sents))
print('відкладених ', len(dev_sents))
print('перевірних  ', len(test_sents))

## 3 · Робоча купа й словник

Повна навчальна частина — це майже вісімдесят тисяч речень, і одне навчання на
ній коштує близько двох хвилин процесорного часу. Нам потрібно **тридцять три**
навчання, тож ми беремо **робочу купу** — перші 16 000 речень навчальної
частини. Це свідоме рішення заради часу, і всі числа зошита стосуються саме її.

Словник будуємо **один раз, по всій робочій купі**, і далі не міняємо. Це
принципово: перплексію не можна порівнювати між моделями з різними словниками,
бо вона рахується відносно словника. Слова, що трапились рідше пʼяти разів,
зводимо в один спецтокен `<unk>` — інакше словник роздувається хвостом із
одноразових слів.

Три спецтокени, які нам потрібні:

- `<pad>` — заповнювач. Речення в батчі різної довжини, і короткі доводиться
  добивати до довжини найдовшого;
- `<eos>` — «кінець речення». Модель має вміти сказати, що фраза скінчилась;
- `<unk>` — «слово поза словником».

In [ ]:
POOL_SIZE = 16000                 # робоча купа речень

# ⚠️ Речення лежать у порядку програм, з яких узяті переклади. Якщо взяти
# «перші 16 000», це буде не менше даних, а вужчий набір програм — і пізніше,
# коли ми ділитимемо купу на частки, менша частка виявилась би ще й вужчою
# за темами. Тому перемішуємо один раз із фіксованим зерном.
shuffler = np.random.default_rng(7)
train_shuffled = [train_sents[i] for i in shuffler.permutation(len(train_sents))]
pool = train_shuffled[:POOL_SIZE]
dev_shuffled = [dev_sents[i] for i in np.random.default_rng(8).permutation(len(dev_sents))]

PAD, EOS, UNK = 0, 1, 2

def build_vocab(sents, min_count=5):
    '''Словник по заданих реченнях: усе, що трапилось >= min_count разів.'''
    counts = Counter(w for s in sents for w in s)
    words = ['<pad>', '<eos>', '<unk>'] + sorted(w for w, n in counts.items()
                                                 if n >= min_count)
    return {w: i for i, w in enumerate(words)}

vocab = build_vocab(pool, 5)

def encode(sents):
    '''Речення -> список номерів слів, у кінці <eos>.'''
    return [[vocab.get(w, UNK) for w in s] + [EOS] for s in sents]

pool_tokens = sum(len(s) + 1 for s in pool)
test_ids = encode(test_sents)
unk_share = np.mean([t == UNK for s in test_ids for t in s])

print('речень у робочій купі', len(pool))
print('слововживань у ній   ', pool_tokens, '(разом із <eos>)')
print('словник              ', len(vocab), 'разом із трьома спецтокенами')
print('частка <unk> у перевірній частині:', round(float(unk_share), 4))

## 4 · Модель і з чого складається її ціна

Мовна модель на трансформері — це чотири речі поспіль:

1. **таблиця ембедингів**: кожному номеру слова відповідає рядок чисел;
2. **позиційне кодування**: окрема таблиця, яка додає до слова інформацію
   «яким за рахунком воно стоїть». Без неї увага не розрізняє порядку;
3. **шари трансформера**: у кожному — self-attention і невелика мережа з двох
   лінійних шарів;
4. **вихідна проєкція**: із вектора розміру `d_model` робить оцінку для
   **кожного** слова словника.

Ми навмисне надрукуємо, скільки ваг у кожній частині. Побачити, де насправді
лежить вага моделі, корисніше, ніж прочитати про це.

**Причинна маска.** Мовна модель не має права підглядати вперед: угадуючи
пʼяте слово, вона мусить бачити лише перші чотири. Тому перед softmax ми
додаємо до оцінок матрицю, у якій усе вище головної діагоналі дорівнює мінус
нескінченності. Після softmax ці позиції дістають рівно нуль ваги.

In [ ]:
class TransformerLM(nn.Module):
    '''Мовна модель: ембединги + позиції + шари трансформера + вихідна проєкція.'''

    def __init__(self, vocab_size, d_model=64, heads=2, layers=2, ff=128,
                 norm_first=True, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, heads, ff, dropout=0.0,
                                           batch_first=True, norm_first=norm_first)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Linear(d_model, vocab_size)
        self.d_model = d_model

    def forward(self, x):
        n = x.shape[1]
        positions = torch.arange(n).unsqueeze(0)
        # множник sqrt(d_model) зрівнює масштаб ембедингів із масштабом позицій
        h = self.emb(x) * math.sqrt(self.d_model) + self.pos(positions)
        # маска забороняє позиції дивитись уперед
        mask = nn.Transformer.generate_square_subsequent_mask(n)
        return self.head(self.enc(h, mask=mask))

CFG = dict(d_model=64, heads=2, layers=2, ff=128)
example = TransformerLM(len(vocab), **CFG)

def count(module):
    return sum(p.numel() for p in module.parameters())

parts = [('ембединги слів', count(example.emb)),
         ('позиційна таблиця', count(example.pos)),
         ('шари трансформера', count(example.enc)),
         ('вихідна проєкція', count(example.head))]
total = count(example)
for name, n in parts:
    print(f'{name:<20} {n:>9}   {100 * n / total:5.1f} %')
print(f'{"разом":<20} {total:>9}')

**Подивись на розподіл.** Сама «розумна» частина — шари з увагою — важить менше,
ніж дві таблиці на краях. Так буває майже завжди, коли словник великий, а модель
маленька: ціну задає **словник**, а не глибина.

## 5 · Увага руками, і чи збігається вона з бібліотечною

Перш ніж міряти вартість уваги, переконаємось, що ми міряємо саме її. Формула
scaled dot-product attention коротка:

`Attention(Q, K, V) = softmax(Q·Kᵀ / √d) · V`

Три матриці — запити `Q`, ключі `K`, значення `V`. Добуток `Q·Kᵀ` дає матрицю
`n × n`: наскільки кожна позиція цікавиться кожною. Ділення на `√d` не дає
оцінкам розлітатись при великому `d`. `softmax` перетворює рядок оцінок на ваги,
які додаються до одиниці. Множення на `V` збирає зважену суму.

Напишемо це пʼятьма рядками й звіримо з бібліотечною реалізацією.

In [ ]:
def attention_by_hand(q, k, v):
    '''Scaled dot-product attention: чотири дії й жодної магії.'''
    scores = q @ k.transpose(-2, -1)          # усі пари позицій
    scores = scores / math.sqrt(q.shape[-1])  # щоб оцінки не розліталися
    weights = torch.softmax(scores, dim=-1)   # рядок ваг, що дають у сумі 1
    return weights @ v                        # зважена сума значень

torch.manual_seed(0)
q, k, v = torch.randn(2, 7, 16), torch.randn(2, 7, 16), torch.randn(2, 7, 16)
ours = attention_by_hand(q, k, v)
library = F.scaled_dot_product_attention(q, k, v)

assert torch.allclose(ours, library, atol=1e-6), 'наша увага розійшлася з бібліотечною!'
print('✅ наша увага збігається з F.scaled_dot_product_attention')
print('   найбільше відхилення:', f'{(ours - library).abs().max().item():.2e}')

# і ось звідки береться квадрат: матриця оцінок має розмір n × n
scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
print('   форма матриці оцінок для 7 позицій:', tuple(scores.shape),
      '— тобто', scores.shape[-1] * scores.shape[-2], 'клітинок на приклад')

## 6 · Замір перший: однакова робота, різна довжина

Тепер головне питання вартості. Ми беремо **сталий обсяг роботи**: добуток
«кількість речень у батчі × довжина речення» тримаємо рівним 8192 у кожному
рядку таблиці. Позицій однаково — міняється лише те, як вони розкладені: чи це
1024 коротких рядки по 8 позицій, чи 8 довгих по 1024.

Поруч ставимо `nn.LSTM` тієї самої ширини. Він читає послідовність крок за
кроком, тож його вартість має бути **лінійною** за довжиною — а отже, при сталій
роботі не мінятись зовсім.

Міряємо один крок навчання: прямий хід, втрата, зворотний хід. Три зерна, беремо
медіану. Один потік, процесорний годинник.

In [ ]:
HIDDEN = 128
TOTAL_POSITIONS = 8192

class AttentionBlock(nn.Module):
    '''Одноголова увага з чотирма проєкціями — щоб ваг було стільки ж порядком.'''
    def __init__(self, hidden):
        super().__init__()
        self.to_q = nn.Linear(hidden, hidden)
        self.to_k = nn.Linear(hidden, hidden)
        self.to_v = nn.Linear(hidden, hidden)
        self.to_out = nn.Linear(hidden, hidden)
        self.scale = 1.0 / math.sqrt(hidden)

    def forward(self, x):
        q, k, v = self.to_q(x), self.to_k(x), self.to_v(x)
        weights = torch.softmax(torch.bmm(q, k.transpose(1, 2)) * self.scale, dim=-1)
        return self.to_out(torch.bmm(weights, v))

def forward_out(model, x):
    '''nn.LSTM повертає пару, наш блок — тензор; зводимо до одного вигляду.'''
    y = model(x)
    return y[0] if isinstance(y, tuple) else y

def step_seconds(make_model, length, batch, seed, reps=2):
    '''Процесорний час одного кроку навчання, усереднений по reps повторах.'''
    torch.manual_seed(seed)
    model = make_model(HIDDEN)
    x = torch.randn(batch, length, HIDDEN)
    forward_out(model, x).sum().backward()      # прогрів: перший виклик дорожчий
    model.zero_grad()
    start = time.process_time()
    for _ in range(reps):
        forward_out(model, x).pow(2).mean().backward()
        model.zero_grad()
    return (time.process_time() - start) / reps

make_lstm = lambda hidden: nn.LSTM(hidden, hidden, batch_first=True)

section = time.process_time()
print('потоків:', torch.get_num_threads(),
      '· батч × довжина =', TOTAL_POSITIONS, '· три зерна, медіана')
print(f"{'довжина':>8}{'батч':>7}{'LSTM, с':>11}{'увага, с':>11}")

timing = {}
for length in (8, 32, 128, 256, 512, 1024):
    batch = TOTAL_POSITIONS // length
    lstm_times = [step_seconds(make_lstm, length, batch, s) for s in (0, 1, 2)]
    attn_times = [step_seconds(AttentionBlock, length, batch, s) for s in (0, 1, 2)]
    # округлюємо ОДРАЗУ: усі відношення далі рахуються з надрукованих чисел
    timing[length] = (round(float(np.median(lstm_times)), 4),
                      round(float(np.median(attn_times)), 4))
    print(f'{length:>8}{batch:>7}{timing[length][0]:>11.4f}{timing[length][1]:>11.4f}')

base_lstm, base_attn = timing[8]
print()
for length in timing:
    print(f'  від довжини 8 до {length:>4}:  LSTM у {timing[length][0] / base_lstm:.2f} раза,'
          f'  увага у {timing[length][1] / base_attn:.2f} раза')
print('процесорний час розділу:', round(time.process_time() - section, 1), 'с')

**Читай так.** Роботи в кожному рядку однаково, а час уваги росте — бо матриця
оцінок має розмір `n × n` на кожен приклад у батчі, і при сталій кількості
позицій довші рядки означають більші матриці. Час `LSTM` при цьому стоїть на
місці: йому байдуже, чи це багато коротких рядків, чи мало довгих.

Зверни увагу, що на найкоротших довжинах увага **дешевша** за `LSTM`. Квадрат
стає проблемою не одразу — він стає проблемою тоді, коли довжина переростає
ширину моделі.

## 7 · Замір другий: скільки памʼяті зʼїдає довжина

Час — не єдина валюта. Матриця оцінок мусить **лежати в памʼяті** цілком, поки
рахується зворотний хід. Її розмір рахується без жодного заміру:

`байтів = батч × голів × n × n × 4`

Чотири байти — розмір одного числа `float32`. Перевіримо формулу, справді
створивши тензор і спитавши в нього розмір.

In [ ]:
BATCH_MEM, HEADS_MEM = 8, 4

def scores_bytes(n, batch=BATCH_MEM, heads=HEADS_MEM):
    '''Скільки важить матриця оцінок уваги для довжини n.'''
    return batch * heads * n * n * 4

# звіряємо формулу з дійсністю: створюємо справжній тензор
probe = torch.zeros(BATCH_MEM, HEADS_MEM, 128, 128, dtype=torch.float32)
real_bytes = probe.numel() * probe.element_size()
assert real_bytes == scores_bytes(128), 'формула розійшлася з дійсністю!'
print('✅ формула збігається з розміром справжнього тензора:', real_bytes, 'байтів')
print()

print(f"{'довжина':>9}{'матриця оцінок':>18}{'у скільки разів більше':>26}")
previous = None
for n in (64, 128, 256, 512, 1024, 2048, 4096):
    mb = scores_bytes(n) / 1024 / 1024
    grow = '—' if previous is None else f'{mb / previous:.2f}× за подвоєння'
    print(f'{n:>9}{mb:>15.2f} МБ{grow:>26}')
    previous = mb

**Ось і вся арифметика довжини контексту.** Кожне подвоєння довжини множить
памʼять під увагу на чотири. Модель, яка спокійно тримає 512 позицій, на 4096
потребує в **64 рази** більше памʼяті саме під цю матрицю — при тому, що
ембединги, шари й вихідна проєкція не змінилися взагалі.

## 8 · Замір третій: кеш ключів і значень

Коли модель **пише** текст, вона робить це по одному слову. Наївний спосіб: щоб
дописати слово номер `t`, подати їй увесь текст від початку й порахувати увагу
для всіх `t` позицій — хоча `t−1` із них порахована на попередньому кроці й
відтоді не змінилась.

**Кеш ключів і значень (KV cache)** — це проста ідея: зберігати `K` і `V`
попередніх позицій і на новому кроці рахувати лише **один** новий рядок уваги —
запит нової позиції проти всіх збережених ключів.

Заміряємо обидва способи для одного шару уваги: скільки коштує написати 256
слів наївно й скільки — з кешем.

In [ ]:
D_GEN, STEPS_GEN = 128, 256

def generate_no_cache(length):
    # Наївно: на кожному кроці рахуємо увагу для ВСЬОГО префікса заново.
    torch.manual_seed(0)
    x = torch.randn(1, length, D_GEN)
    start = time.process_time()
    with torch.no_grad():
        for t in range(1, length + 1):
            prefix = x[:, :t]
            F.scaled_dot_product_attention(prefix, prefix, prefix)
    return time.process_time() - start

def generate_with_cache(length):
    # З кешем: тримаємо K і V у готовому буфері й рахуємо лише рядок нової
    # позиції проти всього, що вже накопичилось.
    torch.manual_seed(0)
    x = torch.randn(1, length, D_GEN)
    keys = torch.zeros(1, length, D_GEN)      # ось він, кеш ключів
    values = torch.zeros(1, length, D_GEN)    # і кеш значень
    start = time.process_time()
    with torch.no_grad():
        for t in range(length):
            step = x[:, t:t + 1]              # лише одна нова позиція
            keys[:, t] = step[:, 0]
            values[:, t] = step[:, 0]
            F.scaled_dot_product_attention(step, keys[:, :t + 1], values[:, :t + 1])
    return time.process_time() - start

section = time.process_time()
naive = round(float(np.median([generate_no_cache(STEPS_GEN) for _ in range(3)])), 4)
cached = round(float(np.median([generate_with_cache(STEPS_GEN) for _ in range(3)])), 4)

print(f'написати {STEPS_GEN} позицій по одній, один шар уваги:')
print(f'  без кешу {naive:.4f} с процесорного часу')
print(f'  з кешем  {cached:.4f} с')
print(f'  з кешем дешевше у {naive / cached:.1f} раза')
print()

# та сама різниця, порахована без годинника: скільки множень робить кожен спосіб
naive_ops = sum(t * t for t in range(1, STEPS_GEN + 1)) * D_GEN
cached_ops = sum(t for t in range(1, STEPS_GEN + 1)) * D_GEN
print('те саме в множеннях, без годинника:')
print(f'  без кешу {naive_ops:>12} — сума квадратів росте як куб довжини')
print(f'  з кешем  {cached_ops:>12} — сума самих t росте як квадрат')
print(f'  відношення: {naive_ops / cached_ops:.1f}')
print()
print('а тепер ціна самого кешу — скільки він важить на кроці t:')
for t in (128, 512, 2048, 8192):
    kv_mb = 2 * t * D_GEN * 4 / 1024 / 1024      # K і V, по чотири байти на число
    print(f'  на кроці {t:>5}: {kv_mb:7.2f} МБ на один шар і одну голову')
print('процесорний час розділу:', round(time.process_time() - section, 1), 'с')

**Що тут важливо побачити.** Кеш прибирає **повторення** роботи, але не прибирає
квадрат: на кроці `t` новий рядок уваги все одно дивиться на `t` ключів, тож
сумарна робота за весь текст лишається квадратичною. Він також не безкоштовний
за памʼяттю — сам кеш росте лінійно з довжиною й живе, доки модель пише.

## 9 · Навчання: однаковий бюджет і чесна зупинка

Далі йдуть навчання, і тут потрібні дві домовленості, без яких увесь наступний
розділ був би брехнею.

**Перша: однакова стеля оновлень.** Ми будемо порівнювати моделі, навчені на
різній кількості даних. Якщо кожній дати «одну епоху», то модель на 1/16 даних
зробить у шістнадцять разів **менше оновлень** — і програє не тому, що їй бракує
тексту, а тому, що їй бракувало кроків. Ми б виміряли не те, що збиралися. Тому
стеля в усіх одна, і модель на маленькій купі просто пройде свої дані багато разів.

**Друга: зупинка за відкладеною вибіркою.** Модель, яка бачить ті самі тисячу
речень удванадцяте, починає їх **запамʼятовувати**: втрата на навчальних даних
падає, а на нових росте. Це називається перенавчанням, і якщо дати такій моделі
доїхати до стелі, ми виміряємо не брак даних, а силу перенавчання.

Тому кожні 50 оновлень ми дивимось на відкладену вибірку й запамʼятовуємо
найкращий стан моделі. Якщо відкладена перестала кращати три заміри поспіль —
спиняємось і повертаємо той найкращий стан. Так **кожна купа даних дістає свій
найкращий результат**, а не покарання за те, що вона мала.

In [ ]:
BATCH = 64
STEP_CEILING = POOL_SIZE // BATCH      # стеля: одна епоха найбільшої купи
CHECK_EVERY = 50                       # як часто заглядати у відкладену
PATIENCE = 3                           # скільки перевірок терпіти без покращення

def make_batches(encoded, batch_size=BATCH, shuffle_seed=None):
    '''Речення близької довжини кладемо в один батч: інакше половина роботи
    йде на заповнювач <pad>. Порядок самих батчів перемішуємо.'''
    order = sorted(range(len(encoded)), key=lambda i: (len(encoded[i]), i))
    groups = [order[s:s + batch_size] for s in range(0, len(order), batch_size)]
    if shuffle_seed is not None:
        gen = np.random.default_rng(shuffle_seed)
        groups = [groups[i] for i in gen.permutation(len(groups))]
    batches = []
    for group in groups:
        rows = [encoded[i] for i in group]
        width = max(len(r) for r in rows)
        block = np.zeros((len(rows), width), dtype=np.int64)   # нулі — це <pad>
        for r, row in enumerate(rows):
            block[r, :len(row)] = row
        batches.append(torch.from_numpy(block))
    return batches

def perplexity(model, batches):
    '''Перплексія: експонента середньої втрати на слово. Заповнювач не рахуємо.'''
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, reduction='sum')
    total, counted = 0.0, 0
    model.eval()
    with torch.no_grad():
        for block in batches:
            logits = model(block[:, :-1])          # вхід — усе, крім останнього
            target = block[:, 1:]                  # ціль — усе, крім першого
            total += loss_fn(logits.reshape(-1, logits.shape[-1]),
                             target.reshape(-1)).item()
            counted += int((target != PAD).sum())
    model.train()
    return math.exp(total / counted)

def train_lm(train_encoded, dev_batches, seed, lr, report_batches=None,
             ceiling=STEP_CEILING, norm_first=True):
    '''Учить до стелі, але спиняється, коли відкладена перестала кращати.
    Повертає перплексію найкращого стану на report_batches (типово — на самій
    відкладеній).'''
    torch.manual_seed(seed)
    model = TransformerLM(len(vocab), **CFG, norm_first=norm_first)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
    best_dev, best_state, best_step = float('inf'), None, 0
    misses, step, epoch, stop = 0, 0, 0, False
    started = time.process_time()
    while step < ceiling and not stop:
        for block in make_batches(train_encoded, shuffle_seed=seed * 1000 + epoch):
            if step >= ceiling:
                break
            step += 1
            logits = model(block[:, :-1])
            loss = loss_fn(logits.reshape(-1, logits.shape[-1]),
                           block[:, 1:].reshape(-1))
            opt.zero_grad()
            loss.backward()
            opt.step()
            if step % CHECK_EVERY == 0:
                current = perplexity(model, dev_batches)
                if current < best_dev:
                    # запамʼятовуємо найкращий стан, щоб потім до нього повернутись
                    best_dev, best_step, misses = current, step, 0
                    best_state = copy.deepcopy(model.state_dict())
                else:
                    misses += 1
                    if misses >= PATIENCE:
                        stop = True
                        break
        epoch += 1
    if best_state is not None:
        model.load_state_dict(best_state)
    target_batches = report_batches if report_batches is not None else dev_batches
    answer = {'ppl': perplexity(model, target_batches), 'dev': best_dev,
              'stopped_at': best_step, 'epochs': epoch,
              'cpu': time.process_time() - started}
    # звільняємо памʼять одразу: зошит навчає три десятки моделей поспіль,
    # і без цього вони накопичуються, поки машині не стане зле
    del model, opt, best_state
    gc.collect()
    return answer

print('розмір батча       ', BATCH)
print('стеля оновлень     ', STEP_CEILING, '— одна епоха купи з', POOL_SIZE, 'речень')
print('перевіряємо кожні  ', CHECK_EVERY, 'оновлень, терпимо', PATIENCE, 'перевірок без покращення')

## 10 · Швидкість навчання: перебираємо сітку на відкладеній частині

Швидкість навчання (learning rate) — це множник, на який множиться крок
оптимізатора. Замалий — модель ледве рухається за відпущений бюджет; завеликий —
крок перескакує мінімум, і навчання або стоїть на місці, або розвалюється.

Правило, за яким ми його добираємо, коштувало курсу окремої граблі:
**сітка має бути широкою, а мінімум мусить лежати всередині неї**. Якщо
найкраще значення виявилось на краю сітки — сітка закоротка, і справжнього
оптимуму ми не бачили.

Дивимось на **відкладену** частину. Перевірна лишається недоторканою.

⚠️ І одразу пастка, у яку легко втрапити: сітку треба проганяти **тим самим
бюджетом**, з яким потім працюватимеш. На короткому бюджеті виграє більший крок
(модель просто не встигає далеко зайти малим), і дібране так значення на повному
бюджеті виявиться завеликим. Тому нижче стеля така сама, як у всіх наступних
навчаннях.

In [ ]:
pool_encoded = encode(pool)
dev_batches = make_batches(encode(dev_shuffled[:1500]), 256)

section = time.process_time()
print(f'перплексія на ВІДКЛАДЕНІЙ частині, три зерна, стеля {STEP_CEILING} оновлень')
print(f"{'lr':>8}{'медіана':>11}{'найгірше':>11}{'найкраще':>11}")

lr_grid = {}
for lr in (0.0005, 0.001, 0.002, 0.004, 0.008, 0.016):
    runs = [train_lm(pool_encoded, dev_batches, seed, lr)['ppl'] for seed in (0, 1, 2)]
    lr_grid[lr] = [round(v, 2) for v in runs]
    print(f'{lr:>8}{np.median(lr_grid[lr]):>11.2f}'
          f'{max(lr_grid[lr]):>11.2f}{min(lr_grid[lr]):>11.2f}')

best_lr = min(lr_grid, key=lambda k: np.median(lr_grid[k]))
edges = (min(lr_grid), max(lr_grid))
print()
print('найкраща швидкість навчання:', best_lr)
print('вона на краю сітки?', 'ТАК — сітку треба розширити' if best_lr in edges
      else 'ні, мінімум усередині — сітці можна вірити')
print('процесорний час розділу:', round(time.process_time() - section, 1), 'с')

## 11 · Лічильники, з якими ми будемо змагатися

Перш ніж будувати головну криву, потрібен суперник — і це не інша нейромережа.

**Згладжена біграма в суміші з уніграмою.** Модель без жодного навчання: вона
просто рахує, скільки разів слово `b` йшло після слова `a`, і скільки разів
слово `b` трапилось узагалі. Дві поправки роблять її працездатною:

- **добавка `k`** до кожного лічильника пари — щоб пара, якої в навчанні не було,
  не діставала нульової ймовірності (а нуль зруйнував би перплексію);
- **суміш із уніграмою** з вагою `λ`: ймовірність слова — це `λ` частин від
  біграми плюс `1 − λ` частин від простої частоти слова.

Обидва числа — `k` і `λ` — теж гіперпараметри, і добирати їх треба **так само
старанно**, як швидкість навчання нейромережі, і на тій самій відкладеній
частині. Інакше ми порівнюватимемо вилизану мережу з недбалою базою, а це не
порівняння.

In [ ]:
class Counts:
    '''Біграма й уніграма, зібрані один раз; k і λ підставляються потім.'''

    def __init__(self, sents):
        self.V = len(vocab)
        self.unigram, self.bigram, self.context = Counter(), Counter(), Counter()
        for ids in encode(sents):
            for first, second in zip(ids, ids[1:]):
                self.bigram[(first, second)] += 1
                self.context[first] += 1
            for token in ids:
                self.unigram[token] += 1
        self.N = sum(self.unigram.values())

    def perplexity(self, pairs, k, lam):
        '''Перплексія на готовому списку пар «попереднє слово -> наступне».'''
        total = 0.0
        for first, second in pairs:
            p_uni = (self.unigram[second] + 1) / (self.N + self.V)
            p_bi = (self.bigram[(first, second)] + k) / (self.context[first] + k * self.V)
            total -= math.log(lam * p_bi + (1 - lam) * p_uni)
        return math.exp(total / len(pairs))

def to_pairs(sents):
    '''Ті самі пари, які передбачає нейромережа: кожне слово за попереднім.'''
    out = []
    for ids in encode(sents):
        out += list(zip(ids, ids[1:]))
    return out

dev_pairs = to_pairs(dev_shuffled[:1500])
test_pairs = to_pairs(test_sents)

K_GRID = (10, 3, 1, 0.3, 0.1, 0.03, 0.01, 0.003, 0.001, 0.0003, 0.0001)
LAMBDA_GRID = (0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 1.0)

def tune_counts(sents):
    '''Перебирає k і λ на відкладеній частині й повертає найкращу пару.'''
    counts = Counts(sents)
    best = None
    for k in K_GRID:
        for lam in LAMBDA_GRID:
            value = counts.perplexity(dev_pairs, k, lam)
            if best is None or value < best[0]:
                best = (value, k, lam)
    return counts, best

section = time.process_time()
counts_full, (dev_best, best_k, best_lambda) = tune_counts(pool)
print('перебрано положень сітки:', len(K_GRID) * len(LAMBDA_GRID))
print('найкраща добавка k      :', best_k)
print('найкраща вага біграми λ :', best_lambda)
print('на відкладеній частині  :', round(dev_best, 2))
print('на перевірній частині   :',
      round(counts_full.perplexity(test_pairs, best_k, best_lambda), 2))
print('процесорний час розділу :', round(time.process_time() - section, 1), 'с')

## 12 · Головна крива: якість від обсягу даних

Тепер сам замір, заради якого писався зошит.

Беремо пʼять купок різного розміру — 1/16, 1/8, 1/4, 1/2 і вся робоча купа.
На кожній навчаємо трансформер **тричі** (три зерна) з однаковою стелею оновлень,
однаковою швидкістю навчання й зупинкою за відкладеною вибіркою. Поруч на тих
самих даних будуємо лічильники, добираючи `k` і `λ` **окремо для кожного
розміру** — бо оптимальне згладжування залежить від того, скільки даних є.

Оголошуємо результат на **перевірній** частині — уперше й востаннє.

⚠️ **Три зерна — компроміс заради часу.** Розкид, порахований по трьох зернах, є
**нижньою оцінкою** розкиду: четверте й пʼяте зерно цілком можуть виявитись
гіршими за всі три. Тому купи в таблиці нижче радше вужчі за справжні, і
твердження на кшталт «різниця є» треба робити обережно.

In [ ]:
test_batches = make_batches(test_ids, 256)
section = time.process_time()

print(f'швидкість навчання {best_lr}, стеля {STEP_CEILING} оновлень, три зерна')
print(f"{'речень':>8}{'слів':>9}{'спинилась':>11}{'трансформер':>13}"
      f"{'купа по зернах':>21}{'лічильники':>12}")

curve = []
for divisor in (16, 8, 4, 2, 1):
    n = len(pool) // divisor
    part = pool[:n]
    part_encoded = encode(part)
    runs = [train_lm(part_encoded, dev_batches, seed, best_lr,
                     report_batches=test_batches) for seed in (0, 1, 2)]
    values = sorted(round(r['ppl'], 2) for r in runs)
    stops = [r['stopped_at'] for r in runs]
    counts_part, (_, k_part, lam_part) = tune_counts(part)
    counts_test = round(counts_part.perplexity(test_pairs, k_part, lam_part), 2)
    words = sum(len(s) + 1 for s in part)
    curve.append({'sents': n, 'words': words, 'nn': values, 'counts': counts_test,
                  'k': k_part, 'lam': lam_part, 'stops': stops})
    print(f'{n:>8}{words:>9}{int(np.median(stops)):>11}{values[1]:>13.2f}'
          f'{f"{values[0]:.2f}…{values[2]:.2f}":>21}{counts_test:>12.2f}')

print('процесорний час розділу:', round(time.process_time() - section, 1), 'с')

Тепер прочитаємо цю таблицю вголос: наскільки кожне подвоєння даних покращує
кожну з двох моделей і чи розрив між ними скорочується.

In [ ]:
print('що дає подвоєння даних (від меншої купи до наступної):')
for previous, current in zip(curve, curve[1:]):
    nn_gain = previous['nn'][1] - current['nn'][1]
    counts_gain = previous['counts'] - current['counts']
    print(f"  {previous['sents']:>5} -> {current['sents']:>5} речень:"
          f"  трансформер {nn_gain:+8.2f}   лічильники {counts_gain:+8.2f}")

print()
print('розрив «трансформер мінус лічильники» на кожній купі:')
for row in curve:
    gap = row['nn'][1] - row['counts']
    who = 'лічильники попереду' if gap > 0 else 'трансформер попереду'
    print(f"  {row['sents']:>5} речень: {gap:+8.2f}   {who}")

## 13 · Куди веде ця крива

Обидві криві на логарифмічній сітці лягають майже на пряму: перплексія падає
приблизно як степінь обсягу даних. Це відомий емпіричний факт, і ми можемо його
використати — обережно.

Підженемо пряму `log(перплексія) = a · log(слів) + b` окремо до кожної моделі й
подивимось, де вони перетнуться. **Це екстраполяція, а не замір**: вона
припускає, що обидві криві поводитимуться так само далеко за межами наших даних,
а такого ніхто не обіцяв. Але вона дає порядок величини — і саме порядок нас
цікавить.

In [ ]:
words = np.array([row['words'] for row in curve], dtype=float)
nn_values = np.array([row['nn'][1] for row in curve], dtype=float)
counts_values = np.array([row['counts'] for row in curve], dtype=float)

nn_slope, nn_intercept = np.polyfit(np.log(words), np.log(nn_values), 1)
c_slope, c_intercept = np.polyfit(np.log(words), np.log(counts_values), 1)

print('нахил кривої (у скільки разів падає перплексія за десятикратне зростання даних):')
print(f'  трансформер: показник {nn_slope:.4f}  ->  у {10 ** (-nn_slope):.2f} раза')
print(f'  лічильники : показник {c_slope:.4f}  ->  у {10 ** (-c_slope):.2f} раза')

if nn_slope < c_slope:
    crossing = math.exp((c_intercept - nn_intercept) / (nn_slope - c_slope))
    print()
    print(f'екстраполяція: криві перетнуться приблизно на {crossing:,.0f} слововживаннях'
          .replace(',', ' '))
    print(f'відношення до нашої робочої купи: ×{crossing / words[-1]:.1f}')
    print()
    # перевірка здорового глузду: чи узгоджується обіцянка прямої з тим,
    # що ми справді заміряли в найправішій точці
    last_gap = curve[-1]['nn'][1] - curve[-1]['counts']
    print('перевірка здорового глузду:')
    print(f'  на останній ЗАМІРЯНІЙ точці розрив ще {last_gap:+.2f} на користь лічильників')
    if crossing <= words[-1] * 2:
        print('  а пряма обіцяє перетин майже там само — отже пряма описує наші точки ПОГАНО.')
        print('  Причина видна на графіку: крива трансформера на логарифмічній сітці')
        print('  не пряма, вона загинається. Вір заміряним точкам, а не продовженню.')
    else:
        print('  пряма обіцяє перетин значно далі — це узгоджується з заміром.')
    print('⚠️ у будь-якому разі це припущення, а не замір: за межами наших пʼятьох')
    print('   точок ніхто не обіцяв, що прямі лишаться прямими')
else:
    print()
    print('трансформер падає не швидше за лічильники — на цих даних')
    print('екстраполювати перетин немає підстав')

## 14 · Скільки все це коштувало

Остання клітинка друкує процесорний час усього зошита — те саме число, яке
стоїть у попередженні на початку. Якщо вони розходяться, вір цьому.

In [ ]:
spent = time.process_time() - notebook_started
print(f'процесорного часу на весь зошит: {spent:.1f} с')
print(f'навантаження машини наприкінці : {os.getloadavg()[0]:.2f}')
print(f'моделей навчено                : {6 * 3 + 5 * 3}')

## Завдання

### 🟢 Рівень 1 — База
Додай до сітки швидкості навчання ще два значення знизу (наприклад `0.0001` і
`0.0002`) і переконайся, що мінімум лишився всередині сітки.
**Зроблено, якщо:** таблиця має вісім рядків, і ти можеш назвати, який рядок
найкращий і чому це не край сітки.

### 🟡 Рівень 2 — Плюс
Побудуй ту саму криву «якість від обсягу даних», але **без** однакового бюджету
оновлень: дай кожній купі рівно одну епоху. Порівняй дві криві.
**Зроблено, якщо:** ти можеш назвати числом, наскільки крива з однією епохою
занижує якість малих купок, і пояснити, чому це не є мірою браку даних.

### 🔴 Рівень 3 — Виклик
Заміряй, як бюджет оновлень впливає на результат: візьми найбільшу купу й
навчи модель на 1/4, 1/2, 1 і 2 бюджети.
**Зроблено, якщо:** ти можеш сказати, чи впирається наша модель у брак даних,
чи в брак кроків — і показати це числами з обома купами.